<a href="https://colab.research.google.com/github/anna030608/DS/blob/master/DATATHON/EDA_%EC%B4%88%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이터 확인 및 불러오기

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_df = pd.read_csv('train.csv')

In [ ]:
print(train_df.shape)

(2556, 19)


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2556 entries, 0 to 2555
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Name                                   2556 non-null   object 
 1   Gender                                 2556 non-null   object 
 2   Age                                    2556 non-null   int64  
 3   City                                   2556 non-null   object 
 4   Working Professional or Student        2556 non-null   object 
 5   Profession                             1883 non-null   object 
 6   Academic Pressure                      502 non-null    float64
 7   Work Pressure                          2054 non-null   float64
 8   CGPA                                   502 non-null    float64
 9   Study Satisfaction                     502 non-null    float64
 10  Job Satisfaction                       2054 non-null   float64
 11  Slee

In [ ]:
test_df.info()

NameError: name 'test_df' is not defined

## columns 설명
- id
- Name
- Gender (Female/Male)
- Age
- City
- Working Professional or Student
- Profession: 직업
- Academic Pressure: 학업 스트레스 수준 (1~5)
- Work Pressure: 업무 스트레스 수준 (1~5)
- CGPA: 학점(성취도 평가)
- Study Satisfaction: 학업 만족도 (1~5)
- Job Satisfaction: 직무 만족도 (1~5)
- Sleep Duration: 수면시간
- Dietary Habits: 식습관 (Healthy/Unhealthy 등)
- Degree: 학위
- Have you ever had suicidal thoughts ?: 자살 생각 경험 여부
- Work/Study Hours: 주당 공부/근무 시간
- Financial Stress: 경제적 스트레스 (1~5)
- Family History of Mental Illness: 가족력 여부
- Depression: 타겟 변수 (0: 우울X / 1: 우울O)
  - 우울감 경험 여부/위험 가능성
  | 구분             | 의미                            |
| -------------- | ----------------------------- |
| Depression = 1 | 설문 기반 우울 위험 또는 우울감 경험 가능성이 있다 |
| Depression = 0 | 그렇지 않다 (우울 위험 낮음)             |
| 실제 진단 여부       | 아님 (임상 진단 아님)                 |


In [ ]:
train_df.head(3)

In [ ]:
train_df.describe()

In [ ]:
print(train_df.isnull().sum())

In [ ]:
print(train_df.isnull().mean())

# Depression 분포 확인하기

In [ ]:
train_df['Depression'].value_counts()

In [ ]:
train_df['Depression'].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='Depression', data=train_df)

total = len(train_df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

plt.title("Depression Distribution (%)")
plt.show()

0 (비우울): 81.8%

1 (우울): 18.2%

- 18%면 약 5명 중 1명은 우울한 상태 -> 꽤 높은 수치

- 엄청 심한 불균형은 아님..
- 하지만, 불균형이 있기 때문에 모델링할 때 class_weight을 줘서 해결하면 될 것 같다.
---

## 직장인 vs 학생별 우울 비교

In [ ]:
# 직장인 vs 학생별 Depression 비율 확인

group_ratio = train_df.groupby('Working Professional or Student')['Depression'].value_counts(normalize=True).unstack()

print("=== Depression Ratio by Group ===")
print(group_ratio)

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x='Working Professional or Student', y='Depression', data=train_df, estimator=lambda x: sum(x)/len(x))
plt.title("Depression Rate: Student vs Working Professional")
plt.ylabel("Depression Rate")
plt.show()

| 그룹                   | 비우울 (0) | 우울 (1)     |
| -------------------- | ------- | ---------- |
| Student              | 41.45%  | **58.55%** |
| Working Professional | 91.82%  | **8.18%**  |

- 우울증의 중심 집단은 "학생"
- 전체 우울증 평균 18%는 직장인이 80% 이상 차지하기 때문에 낮아진 것 -> 실제로는 학생 집단에서 우울증일 확률이 매우 높다.

# 다른 변수 분포 확인하기

#### 1. Sleep Duration vs Depression

In [ ]:
sleep_ratio = train_df.groupby('Sleep Duration')['Depression'].mean().sort_values()

print("=== Sleep Duration별 Depression 비율 ===")
print((sleep_ratio * 100).round(2))

In [ ]:
plt.figure(figsize=(8,5))
sleep_ratio.sort_values().plot(kind='bar')
plt.title("Depression Rate by Sleep Duration")
plt.ylabel("Depression Rate")
plt.show()

- 정상적인 수면 데이터가 아님.
- 전처리 필요

#### 1-1. 정상적인 범위의 수면 카테고리 필터링 + 시각화

In [ ]:
print("=== Sleep Duration 고유값 개수 ===")
print(train_df['Sleep Duration'].nunique())

print("\n=== Sleep Duration 상위 30개 값 ===")
print(train_df['Sleep Duration'].value_counts().head(30))

In [ ]:
# 정상 수면 카테고리만 필터링
valid_sleep = [
    'Less than 5 hours',
    '5-6 hours',
    '7-8 hours',
    'More than 8 hours'
]

sleep_clean_df = train_df[train_df['Sleep Duration'].isin(valid_sleep)]

sleep_ratio_clean = sleep_clean_df.groupby('Sleep Duration')['Depression'].mean()

print("=== Sleep Duration별 Depression 비율 ===")
print((sleep_ratio_clean * 100).round(2))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='Sleep Duration', y='Depression', data=sleep_clean_df,
            order=valid_sleep)
plt.title("Depression Rate by Sleep Duration (Cleaned)")
plt.show()

| 수면 시간             | 우울 비율      |
| ----------------- | ---------- |
| Less than 5 hours | **23.47%** |
| 5-6 hours         | 16.56%     |
| 7-8 hours         | 17.83%     |
| More than 8 hours | **13.87%** |

- 5시간 미만 수면 → 우울 위험 가장 높음 (23.5%)
- 8시간 초과 수면 → 우울 위험 가장 낮음 (13.9%)

- 최대-최소 차이 : 약 9.6%

=> 수면 부족 집단은 충분한 수면 집단 대비 약 1.7배 높은 우울 위험을 보인다.

(“수면 부족은 우울 위험과 강한 연관성을 보인다.”?)

**수면 5시간 미만 집단은 가장 높은 우울 위험을 보인다**

#### 2. Financial Stress vs Depression

In [ ]:
financial_ratio = train_df.groupby('Financial Stress')['Depression'].mean()

print("=== Financial Stress별 Depression 비율 ===")
print((financial_ratio * 100).round(2))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x='Financial Stress', y='Depression', data=train_df)
plt.title("Depression Rate by Financial Stress")
plt.show()

| Financial Stress | 우울 비율     |
| ---------------- | --------- |
| 1                | 8.5%      |
| 2                | 10.1%     |
| 3                | 17.4%     |
| 4                | 22.0%     |
| 5                | **33.3%** |

- 재정 스트레스가 가장 높은 집단은
가장 낮은 집단보다 거의 4배 높은 우울 위험을 보인다.

=> 경제적 스트레스가 증가할수록, 우울 위험이 선형적으로 증가함

(= 재정 스트레스는 우울 위험을 설명하는 가장 강력한 요인이다. ??)

#### 3. Work/Study Hours vs Depression

In [ ]:
hours_ratio = train_df.groupby('Work/Study Hours')['Depression'].mean()

print("=== Work/Study Hours별 Depression 비율 ===")
print((hours_ratio * 100).round(2))

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(x='Work/Study Hours', y='Depression', data=train_df)
plt.title("Depression Rate by Work/Study Hours")
plt.show()

- 0-5시간 → 낮은 우울 (8~13%)
- 6-8시간 → 급상승 (18~25%)
- 10-12시간 → 매우 높음 (30% 이상)

=> 장시간 학업/근무는 우울 위험과 강한 연관성을 보임.

#### 4. Work Pressure

In [ ]:
working_df = train_df[train_df['Working Professional or Student'] == 'Working Professional']

work_pressure_ratio = working_df.groupby('Work Pressure')['Depression'].mean()

print("=== Work Pressure별 Depression 비율 (직장인) ===")
print((work_pressure_ratio * 100).round(2))

In [ ]:
sns.barplot(x='Work Pressure', y='Depression', data=working_df)
plt.title("Depression Rate by Work Pressure (Working)")
plt.show()

| Work Pressure | 우울 비율      |
| ------------- | ---------- |
| 1             | 2.25%      |
| 2             | 3.23%      |
| 3             | 5.44%      |
| 4             | 10.48%     |
| 5             | **19.61%** |

- 단조 증가.
- 압박감이 증가할수록 우울 위험 증가
- 1 -> 5단계에서 약 8.7배 증가함.

#### 5. Academic Pressure

In [ ]:
student_df = train_df[train_df['Working Professional or Student'] == 'Student']

academic_ratio = student_df.groupby('Academic Pressure')['Depression'].mean()

print("=== Academic Pressure별 Depression 비율 (학생) ===")
print((academic_ratio * 100).round(2))

In [ ]:
sns.barplot(x='Academic Pressure', y='Depression', data=student_df)
plt.title("Depression Rate by Academic Pressure (Student)")
plt.show()

| Academic Pressure | 우울 비율      |
| ----------------- | ---------- |
| 1                 | 19.41%     |
| 2                 | 37.48%     |
| 3                 | 60.16%     |
| 4                 | 76.14%     |
| 5                 | **86.09%** |

- 학업 압박 5단계 학생의 86%가 우울 상태임

=> 학생 집단에서 Academic Pressure는 거의 우울을 설명하는 핵심 요인처럼 보인다.

#### 6. Job Satisfaction

In [ ]:
job_ratio = working_df.groupby('Job Satisfaction')['Depression'].mean()

print("=== Job Satisfaction별 Depression 비율 (직장인) ===")
print((job_ratio * 100).round(2))

In [ ]:
sns.barplot(x='Job Satisfaction', y='Depression', data=working_df)
plt.title("Depression Rate by Job Satisfaction")
plt.show()

| Job Satisfaction | 우울 비율  |
| ---------------- | ------ |
| 1                | 17.40% |
| 2                | 9.83%  |
| 3                | 5.45%  |
| 4                | 3.94%  |
| 5                | 3.88%  |

- 단조 감소 형태

-> 직무 만족도가 올라갈수록 우울 위험이 급격히 감소한다.

#### 7. Study Satisfaction

In [ ]:
study_ratio = student_df.groupby('Study Satisfaction')['Depression'].mean()

print("=== Study Satisfaction별 Depression 비율 (학생) ===")
print((study_ratio * 100).round(2))

In [ ]:
sns.barplot(x='Study Satisfaction', y='Depression', data=student_df)
plt.title("Depression Rate by Study Satisfaction")
plt.show()

| Study Satisfaction | 우울 비율  |
| ------------------ | ------ |
| 1                  | 70.76% |
| 2                  | 64.54% |
| 3                  | 57.60% |
| 4                  | 51.33% |
| 5                  | 47.22% |

- 단조 감소형태.
- 학업 만족도가 높을수록 우울 위험 감소

=> 학생과 직장인은 우울함 리스크 구조가 다르다.

---


In [ ]:
print(len(working_df))
print(len(student_df))

### Q. (직장인) 경제적 스트레스가 높은 사람이 장시간 근무까지 하면 우울 위험이 폭발적으로 증가하는가?

In [ ]:
## 직장인 데이터 분리
working_df = train_df[train_df['Working Professional or Student'] == 'Working Professional']

In [ ]:
pivot_fs_hours = working_df.pivot_table(
    values='Depression',
    index='Financial Stress',
    columns='Work/Study Hours',
    aggfunc='mean'
)

print("=== Financial Stress × Work Hours Depression 비율 ===")
print((pivot_fs_hours * 100).round(2))

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(pivot_fs_hours, annot=True, fmt=".2f", cmap="Reds")
plt.title("Depression Rate: Financial Stress × Work Hours (Working)")
plt.show()

- Low financial Stress 집단 : 시간이 늘어나도 우울 위험은 크게 안 오름.
- High financial Stress 집단: 시간이 늘어나면 우울 위험이 폭발적으로 증가함.

=> 장시간 노동 자체가 문제라기보단, 경제적 압박이 있는 상태에서의 장시간 노동이 치명적
(상호작용 효과 interaction effect)

### Q. (학생) 학업 압박이 높은 상태에서 수면이 부족하면 우울 위험이 폭발적으로 증가하는가?

In [ ]:
# 학생 데이터 분리
student_df = train_df[train_df['Working Professional or Student'] == 'Student']

In [ ]:
valid_sleep = [
    'Less than 5 hours',
    '5-6 hours',
    '7-8 hours',
    'More than 8 hours'
]

student_df_clean = student_df[student_df['Sleep Duration'].isin(valid_sleep)]

In [ ]:
pivot_ap_sleep = student_df_clean.pivot_table(
    values='Depression',
    index='Academic Pressure',
    columns='Sleep Duration',
    aggfunc='mean'
)

print("=== Academic Pressure × Sleep Duration Depression 비율 ===")
print((pivot_ap_sleep * 100).round(2))

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(pivot_ap_sleep, annot=True, fmt=".2f", cmap="Reds")
plt.title("Depression Rate: Academic Pressure × Sleep (Student)")
plt.show()

- Academic Pressure 내에서 수면 시간을 보면, 수면 부족일 때가 가장 높음.
- 압박이 극단적으로 높으면, 수면 부족은 추가적으로 우울감 위험을 높인다.

=> 압박이 낮으면 수면 영향은 제한적

=> 압박이 높으면 수면 부족이 리스크를 더 증폭

---

### Q. 만족도가 높으면 압박의 부정적 효과를 완충하는가?

#### - 직장인

In [ ]:
pivot_wp_js = working_df.pivot_table(
    values='Depression',
    index='Work Pressure',
    columns='Job Satisfaction',
    aggfunc='mean'
)

print("=== Work Pressure × Job Satisfaction Depression 비율 ===")
print((pivot_wp_js * 100).round(2))

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(pivot_wp_js, annot=True, fmt=".2f", cmap="Reds")
plt.title("Depression Rate: Work Pressure × Job Satisfaction (Working)")
plt.show()

- Work Pressure = 5 (압박 최고 단계)
| Job Satisfaction | 우울 비율  |
| ---------------- | ------ |
| 1                | 36.76% |
| 5                | 9.21%  |

-> 같은 직장 압밥 수준에서도 만족도가 높으면, 우울 위험이 4배 이상 감소한다.

= Job Satisfaction이 그 위험을 완충한다.

#### - 학생

In [ ]:
pivot_ap_ss = student_df.pivot_table(
    values='Depression',
    index='Academic Pressure',
    columns='Study Satisfaction',
    aggfunc='mean'
)

print("=== Academic Pressure × Study Satisfaction Depression 비율 ===")
print((pivot_ap_ss * 100).round(2))

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(pivot_ap_ss, annot=True, fmt=".2f", cmap="Reds")
plt.title("Depression Rate: Academic Pressure × Study Satisfaction (Student)")
plt.show()

| Study Satisfaction | 우울 비율  |
| ------------------ | ------ |
| 1                  | 91.53% |
| 5                  | 77.86% |

- 압박이 극단적으로 높으면, 만족도가 높아도 우울 위험은 매우 높다.

직장인에서는
- 압박이 높아도 만족도가 높으면 상당히 완화됨.

반면, 학생 입장에서는
- 압박이 너무 높으면 만족도도 충분히 완화하지 못한다.

---
인사이트

**=> 직장인은 소속감/의미감 등이 보호 요인이다?**

**=> 학생은 "압박 수준 자체를 낮추는 것"이 더 중요하다?**

#### 전체 구조 요약

직장인 우울 구조
- 경제적 압박
  - 장시간 노동
  - 업무 압박

  -> 우울 위험 증가

  -> 직무 만족도가 완충...

학생 우울 구조
- 학업 압박
  -> 결정적인 리스크..

  -> 수면 부족이 증폭시킴.

  -> 만족도는 일부 완충 효과

---

# 1차 통계적 검정

#### 1. Academic Pressure ↔ Depression

In [ ]:
from scipy.stats import chi2_contingency

# 학생 데이터
student_df = train_df[train_df['Working Professional or Student'] == 'Student']

# Academic Pressure 결측 제거
student_ap = student_df[['Academic Pressure', 'Depression']].dropna()

In [ ]:
contingency_table = pd.crosstab(
    student_ap['Academic Pressure'],
    student_ap['Depression']
)

print("=== Contingency Table ===")
print(contingency_table)

In [ ]:
## 카이제곱 검정

chi2, p, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)

Chi-square = 6426.7

df = 4

p-value = 0.0
---
- p-value = 0.0 : “Academic Pressure와 Depression은 독립이다”
라는 가설을 완전히 기각한다는 뜻이다.
- Chi-square = 6426.7 : 효과 크기 자체가 매우 큼

=> Academic Pressure는 우울과 통계적으로 매우 강한 관련이 있다.

**=> 카이제곱 검정 결과, 학업 압박과 우울 간 관계는 통계적으로 매우 유의미하였다 (χ²=6426.7, p<0.001)**

#### 2. Study Satisfaction ↔ Depression

- 학생 구조가 일관된지 확인하기 위한 검정

In [ ]:
# Study Satisfaction 결측 제거
student_ss = student_df[['Study Satisfaction', 'Depression']].dropna()

contingency_ss = pd.crosstab(
    student_ss['Study Satisfaction'],
    student_ss['Depression']
)

chi2_ss, p_ss, dof_ss, expected_ss = chi2_contingency(contingency_ss)

print("Chi-square statistic:", chi2_ss)
print("Degrees of freedom:", dof_ss)
print("p-value:", p_ss)

χ² = 793.86

df = 4

p-value ≈ 1.64e-170

---
- p-value가 1.64e-170: Study Satisfaction과 Depression은 독립이 아니다.

-> 학업 만족도는 우울과 통계적으로 매우 강한 관련이 있다.

| 변수                 | χ²   |
| ------------------ | ---- |
| Academic Pressure  | 6426 |
| Study Satisfaction | 794  |

=> 압박이 훨씬 강한 요인임

---

#### 3. Work Pressure ↔ Depression

In [ ]:
# 직장인 데이터
working_df = train_df[train_df['Working Professional or Student'] == 'Working Professional']

working_wp = working_df[['Work Pressure', 'Depression']].dropna()

contingency_wp = pd.crosstab(
    working_wp['Work Pressure'],
    working_wp['Depression']
)

chi2_wp, p_wp, dof_wp, expected_wp = chi2_contingency(contingency_wp)

print("Chi-square statistic:", chi2_wp)
print("Degrees of freedom:", dof_wp)
print("p-value:", p_wp)

χ² = 6086.46

df = 4

p-value = 0.0

---
- p-value = 0.0 : 독립 가설 기각

-> Work Pressure는 우울과 매우 강한 통계적 관련이 있다.

| 그룹  | 주요 압박 변수          | χ²   |
| --- | ----------------- | ---- |
| 학생  | Academic Pressure | 6426 |
| 직장인 | Work Pressure     | 6086 |


=> 두 집단 모두 Pressure 가 핵심 요인이다.

---

#### 4. Financial Stress ↔ Depression

In [ ]:
working_fs = working_df[['Financial Stress', 'Depression']].dropna()

contingency_fs = pd.crosstab(
    working_fs['Financial Stress'],
    working_fs['Depression']
)

chi2_fs, p_fs, dof_fs, expected_fs = chi2_contingency(contingency_fs)

print("Chi-square statistic:", chi2_fs)
print("Degrees of freedom:", dof_fs)
print("p-value:", p_fs)

χ² = 4519.36

df = 4

p-value = 0.0

---
- p-value = 0.0 : 독립 가설 기각

-> Financial Stress는 우울과 통계적으로 매우 강한 관련이 있다.

##### 직장인 - 변수 강도 비교

| 변수               | χ²   |
| ---------------- | ---- |
| Work Pressure    | 6086 |
| Financial Stress | 4519 |


- 둘 다 매우 강함.
- 하지만, 위에서 확인해본 바로는 Financial Stress가 증폭의 중심이었음..

---

# 추가 피처 확인하기

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_categorical_risk(base_df, col, target='Depression', top_n=None):
    # 0) 분석용 df: 필요한 컬럼만 떼고 결측 제거
    df = base_df[[col, target]].dropna()

    print(f"\n==================== {col} ====================")

    # 1) 분포(Count)
    counts = df[col].value_counts()
    print("\n=== 분포(Count) ===")
    print(counts)

    plt.figure(figsize=(7,4))
    sns.countplot(data=df, x=col, order=counts.index)
    plt.title(f"Count (Histogram) of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha='right')
    plt.show()

    # 2) 우울 위험 비율(%) (Depression=1 평균)
    rate_df = (
        df.groupby(col)[target]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={target: 'Depression Rate (%)'})
    )

    # 표본수
    rate_df['N'] = rate_df[col].map(counts)

    print("\n=== 우울 비율(%) ===")
    print(rate_df.sort_values('Depression Rate (%)', ascending=False))

    # 3) 비율 시각화
    plt.figure(figsize=(7,4))
    sns.barplot(
        data=rate_df.sort_values('Depression Rate (%)', ascending=False),
        x=col, y='Depression Rate (%)'
    )
    plt.title(f"Depression Rate by {col}")
    plt.ylim(0, 100)
    plt.xlabel(col)
    plt.ylabel("Depression Rate (%)")
    plt.xticks(rotation=30, ha='right')
    plt.show()

    # 4) 카이제곱 검정
    ct = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = chi2_contingency(ct)

    print("\n=== Chi-square test ===")
    print("Chi-square:", chi2)
    print("dof:", dof)
    print("p-value:", p)

    # 5) 효과크기: Cramér's V
    n = ct.values.sum()
    r, k = ct.shape
    cramers_v = np.sqrt((chi2 / n) / (min(r-1, k-1)))
    print("Cramér's V:", cramers_v)

    return rate_df, ct

#### 1. Suicidal Thoughts vs Depression

In [ ]:
rate_df_A, ct_A = analyze_categorical_risk(
    base_df=train_df,
    col='Have you ever had suicidal thoughts ?'
)

- 분포
| 자살 생각 경험 | N      |
| -------- | ------ |
| No       | 71,138 |
| Yes      | 69,562 |

-> 표본 균형 O -> 신뢰 가능.

- 우울 비율 비교
| 자살 생각 경험 | 우울 비율  |
| -------- | ------ |
| No       | 4.86%  |
| Yes      | 31.78% |

-> 약 6.5배 차이 (매우 큼)

카이제곱 검정

- χ² = 17142
- p-value = 0.0

→ 통계적으로 완전히 유의

---
- Cramér’s V = 0.349 : 중간 ~ 꽤 강한 효과 크기

-> 실질적으로 큰 차이

---
**자살 생각 경험 여부는 우울과 매우 강한 관련을 보였다 (χ²=17142, p<0.001, Cramér’s V=0.35).**

**하지만 이는 우울의 원인이라기보다는 동반 지표일 가능성이 높다.**

#### 2. Family History

In [ ]:
rate_df_B, ct_B = analyze_categorical_risk(
    base_df=train_df,
    col='Family History of Mental Illness'
)

- 가족력 분포
| 가족력 | N      |
| --- | ------ |
| No  | 70,758 |
| Yes | 69,942 |

-> 표뵨 균형 좋음 -> 신뢰도 O

---
- 우울 비율 비교.
| 가족력 | 우울 비율  |
| --- | ------ |
| Yes | 18.81% |
| No  | 17.54% |

-> 차이 약 1.27% (매우 작음)

---

카이제곱 검정

- χ² = 38.23
- p-value ≈ 6e-10

→ 통계적으로는 유의하다.

- Cramér’s V = 0.016 : 효과 크기 매우 작음
---

**표본이 너무 커서 유의해진 케이스 => 가족력은 주요 위험 요인으로 보기 어렵다**

#### 3. Gender

In [ ]:
rate_df_C, ct_C = analyze_categorical_risk(
    base_df=train_df,
    col='Gender'
)

- Gender 분포
| Gender | N      |
| ------ | ------ |
| Male   | 77,464 |
| Female | 63,236 |

-> 표본 충분. 균형도 나쁘지 않

---

- 우울 비율 비교
| Gender | 우울 비율  |
| ------ | ------ |
| Male   | 18.46% |
| Female | 17.82% |

-> 차이 약 0.63% (거의 차이 없음.)

---

카이제곱 검정

- χ² = 9.29
- p = 0.0023

-> 통계적으로는 유의.

- Cramér’s V = 0.008 : 사실상 효과는 없다고 보면 된다.

**=> 이 데이터에서는 성별은 우울 위험을 설명하는 주요 요인이 아니다.**

#### 4. Dietary Habits (식습관)

In [ ]:
train_df['Dietary Habits'].value_counts()

In [ ]:
col = 'Dietary Habits'
target = 'Depression'

## 필터링
valid_categories = ['Healthy', 'Moderate', 'Unhealthy']

df = train_df[[col, target]].dropna()
df = df[df[col].isin(valid_categories)]

In [ ]:
counts = df[col].value_counts()

print("=== 분포(Count) ===")
print(counts)

plt.figure(figsize=(7,4))
sns.countplot(data=df, x=col, order=counts.index)
plt.title("Count (Histogram) of Dietary Habits")
plt.xlabel("Dietary Habits")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.show()

In [ ]:
rate_df = (
    df.groupby(col)[target]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={target: 'Depression Rate (%)'})
)

rate_df['N'] = rate_df[col].map(counts)

print("\n=== 우울 비율(%) + 표본수(N) ===")
print(rate_df.sort_values('Depression Rate (%)', ascending=False))

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(
    data=rate_df.sort_values('Depression Rate (%)', ascending=False),
    x='Dietary Habits',
    y='Depression Rate (%)'
)

plt.title("Depression Rate by Dietary Habits")
plt.ylim(0, 100)
plt.xticks(rotation=30)
plt.show()

In [ ]:
ct = pd.crosstab(df[col], df[target])
chi2, p, dof, expected = chi2_contingency(ct)

print("\n=== Chi-square test ===")
print("Chi-square:", chi2)
print("dof:", dof)
print("p-value:", p)

# 효과 크기
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt((chi2 / n) / (min(r-1, k-1)))

print("Cramér's V:", cramers_v)

 - Dietary Habits 분포
 | 식습관       | N      |
| --------- | ------ |
| Moderate  | 49,705 |
| Unhealthy | 46,227 |
| Healthy   | 44,741 |

-> 표본 충분, 균형O -> 신뢰도 O

---

- 우울 비율 비교
| 식습관       | 우울 비율      |
| --------- | ---------- |
| Unhealthy | **26.05%** |
| Moderate  | 16.56%     |
| Healthy   | **11.80%** |

-> Healthy와 Unhealthy 차이 : 약 14.2% (꽤 큼)

---

카이제곱 검정

- χ² = 3238
- p = 0.0

-> 유의미함.

- Cramér’s V = 0.152 : 약하지만, 의미 있는 수준

**=> 생활습관은 우울 위험과 유의미한 관련이 있다.**

#### 5. Age

In [ ]:
col = 'Age'
target = 'Depression'

df = train_df[[col, target]].dropna()

In [ ]:
print("=== Age 기본 통계 ===")
print(df[col].describe())

plt.figure(figsize=(6,4))
sns.histplot(data=df, x=col, bins=20, kde=True)
plt.title("Distribution of Age")
plt.show()

In [ ]:
print("\n=== 우울 여부별 평균 Age ===")
print(df.groupby(target)[col].mean())

In [ ]:
bins = [18, 25, 35, 45, 55, 65]
labels = ['18-25', '26-35', '36-45', '46-55', '56-65']

df['Age Group'] = pd.cut(df[col], bins=bins, labels=labels, right=False)

# 비율 계산
rate_df = (
    df.groupby('Age Group')[target]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={target: 'Depression Rate (%)'})
)

counts = df['Age Group'].value_counts()
rate_df['N'] = rate_df['Age Group'].map(counts)

print("\n=== 연령대별 우울 비율 ===")
print(rate_df)

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(
    data=rate_df,
    x='Age Group',
    y='Depression Rate (%)'
)

plt.title("Depression Rate by Age Group")
plt.ylim(0, 100)
plt.show()

In [ ]:
## t-test

from scipy.stats import ttest_ind

age_dep = df[df[target] == 1][col]
age_non = df[df[target] == 0][col]

t_stat, p_val = ttest_ind(age_dep, age_non)

print("\n=== t-test ===")
print("t-statistic:", t_stat)
print("p-value:", p_val)

- 평균 Age 비교
| 그룹    | 평균 나이  |
| ----- | ------ |
| 우울 없음 | 43.68세 |
| 우울 있음 | 25.55세 |

-> 차이가 약 18세

---

- 연령대별 우울 비율
| 연령대   | 우울 비율      |
| ----- | ---------- |
| 18–25 | **61.75%** |
| 26–35 | 41.55%     |
| 36–45 | 3.50%      |
| 46–55 | 1.24%      |
| 56–65 | 0.25%      |

-> 35세 이전과 이후가 완전히 다름.

-> 36세 이상부터 급격히 감소

---

t-test 검정
- t = -256
- p = 0.0

-> 통계적으로 완전 유의함. (표본이 커서 p=0은 당연하지만, 평균 차이 자체가 매우 크다)

**=> 우울은 젊은 연령층에서 압도적으로 높다**

**Age가 Student/Working 그룹과 강하게 얽혀 있을 가능성 매우 높음**

#### 5-1. Age - Student/Working 분포

In [ ]:
pd.crosstab(train_df['Working Professional or Student'],
            pd.cut(train_df['Age'], bins=[18,25,35,45,55,65]))

- 36세 이상은 거의 전부 Working
- Student는 거의 35세 이하에 몰려 있음

**=> Age 효과와 Student/Working 효과가 강하게 얽혀 있음**

#### Q. 젊은 사람이라서 위험한 건지? 아니면, 학생이어서 위험한 건지?

#### 5-2. Student 내부 Age 효과

In [ ]:
student_df = train_df[train_df['Working Professional or Student'] == 'Student']

student_df['Age Group'] = pd.cut(
    student_df['Age'],
    bins=[18,25,35,45,55,65]
)

student_rate = (
    student_df.groupby('Age Group')['Depression']
    .mean()
    .mul(100)
)

print("=== Student 내부 연령대별 우울 비율 ===")
print(student_rate)

#### 5-3. Working 내부 Age 효과

In [ ]:
working_df = train_df[train_df['Working Professional or Student'] == 'Working Professional']

working_df['Age Group'] = pd.cut(
    working_df['Age'],
    bins=[18,25,35,45,55,65]
)

working_rate = (
    working_df.groupby('Age Group')['Depression']
    .mean()
    .mul(100)
)

print("=== Working 내부 연령대별 우울 비율 ===")
print(working_rate)

Q. 젊은 사람이라서 위험한 건지? 아니면, 학생이어서 위험한 건지?

- 학생
| 연령    | 우울 비율     |
| ----- | --------- |
| 18–25 | **66.2%** |
| 26–35 | 49.7%     |
| 36–45 | 32.1%     |
| 46–55 | 25.0%     |

-> 학생 내부에서도 나이가 낮을수록 위험이 매우 높다.

- 직장인
| 연령    | 우울 비율 |
| ----- | ----- |
| 18–25 | 47.0% |
| 26–35 | 20.3% |
| 36–45 | 3.3%  |
| 46–55 | 1.1%  |
| 56–65 | 0.2%  |

-> 같은 패턴..

**=> Age 효과는 직업구조와는 무관하게 존재한다.**

**Age 효과는 독립적으로 매우 강력한 변수**

#### 5-4. Age × Academic Pressure

In [ ]:
# Student만 추출
student_df = train_df[train_df['Working Professional or Student'] == 'Student'].copy()

# Age 구간화
student_df['Age Group'] = pd.cut(
    student_df['Age'],
    bins=[18,25,35,45,55,65]
)

# Academic Pressure 구간화 (High vs Low로 단순화)
student_df['Pressure Level'] = student_df['Academic Pressure'].apply(
    lambda x: 'High (4-5)' if x >= 4 else 'Low/Medium (1-3)'
)

# 결측 제거
student_df_clean = student_df[['Age Group','Pressure Level','Depression']].dropna()

# 그룹별 우울 비율
ap_age_rate = (
    student_df_clean
    .groupby(['Age Group','Pressure Level'])['Depression']
    .mean()
    .mul(100)
    .reset_index()
)

print("=== Age × Academic Pressure 우울 비율 ===")
print(ap_age_rate)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=ap_age_rate,
    x='Age Group',
    y='Depression',
    hue='Pressure Level'
)
plt.title("Age × Academic Pressure Depression Rate")
plt.ylabel("Depression Rate (%)")
plt.show()

- 18-25 학생
| Pressure   | 우울 비율      |
| ---------- | ---------- |
| High       | **86.99%** |
| Low/Medium | 51.46%     |

=> 20대 초반 + 높은 학업 압박 -> 우울 위험 87%

- 25-35 학생
| Pressure   | 우울 비율  |
| ---------- | ------ |
| High       | 75.35% |
| Low/Medium | 33.17% |

=> 압박이 위험을 2배 이상 증폭

- 35세 이상 학생

=> 표본이 매우 작음 (거의 없음)

---

**=> 청년 학생 집단에서 학업적 압박은 결정적인 증폭 요인이다.**

## 직업 압박 X AGE


-

#### 5-5. Age × Financial Stress (전체 집단)

In [ ]:
df = train_df[['Age','Financial Stress','Depression']].dropna()

df['Age Group'] = pd.cut(
    df['Age'],
    bins=[18,25,35,45,55,65]
)

df['Financial Level'] = df['Financial Stress'].apply(
    lambda x: 'High (4-5)' if x >= 4 else 'Low/Medium (1-3)'
)

fs_age_rate = (
    df.groupby(['Age Group','Financial Level'])['Depression']
    .mean()
    .mul(100)
    .reset_index()
)

print("=== Age × Financial Stress 우울 비율 ===")
print(fs_age_rate)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=fs_age_rate,
    x='Age Group',
    y='Depression',
    hue='Financial Level'
)
plt.title("Age × Financial Stress Depression Rate")
plt.ylabel("Depression Rate (%)")
plt.show()

- 18-25 전체
| 재정 스트레스    | 우울 비율  |
| ---------- | ------ |
| High       | 74.12% |
| Low/Medium | 44.17% |

=> 압박보단 조금 약하지만, 강함

- 25-35
| 재정 스트레스    | 우울 비율  |
| ---------- | ------ |
| High       | 52.04% |
| Low/Medium | 25.25% |

=> 재정 스트레스는 청년층에서 명확한 위험 증폭 요인이다.

- 35세 이상.

=> 우울 비율 자체가 매우 낮음.

=> 스트레스 영향도 약해짐.

---

**-  재정 스트레스는 청년층에서만 강하게 작동한다.**

**- 중장년층 : 재정 스트레스가 있어도 우울 위험이 거의 낮음**

**=> 재정 스트레스는 연령에 따라 영향력이 달라진다.**

#### 6. Profession

In [ ]:
train_df['Profession'].nunique()

In [ ]:
train_df['Profession'].unique()

In [ ]:
train_df['Profession'].value_counts().head(10)

In [ ]:
prof_df = train_df[['Profession','Depression']].dropna()

top_prof = prof_df['Profession'].value_counts().head(10).index

prof_df = prof_df[prof_df['Profession'].isin(top_prof)]

prof_rate = (
    prof_df.groupby('Profession')['Depression']
    .mean()
    .mul(100)
    .reset_index()
    .sort_values(by='Depression', ascending=False)
)

print("=== 상위 직업별 우울 비율 ===")
print(prof_rate)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=prof_rate,
    y='Profession',
    x='Depression'
)

plt.title("Depression Rate by Profession (Top 10)")
plt.xlabel("Depression Rate (%)")
plt.show()

**분석에 앞서, Profession은 거의 전부 Working 집단임 -> 우울 위험이 낮은 직장인에 대한 내부 비교임.**

- 직업간 차이는 존재함.
- 하지만, 최대치도 11% 수준으로, Student(60~80%)에 비하면 매우 낮은 수준.


# 로지스틱 회귀

#### Q. 모든 변수를 동시에 넣었을 때, 어떤 변수가 독립적으로 가장 강한가?

In [ ]:
train_df.isna().mean()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

cols = [
    'Age',
    'Financial Stress',
    'Work Pressure',
    'Academic Pressure',
    'Work/Study Hours',
    'Dietary Habits',
    'Have you ever had suicidal thoughts ?',
    'Working Professional or Student',
    'Depression'
]

model_df = train_df[cols].copy()

# 1) Dietary 정제 (이상 카테고리 제거)
valid_diet = ['Healthy', 'Moderate', 'Unhealthy']
model_df = model_df[model_df['Dietary Habits'].isin(valid_diet)]

# 2) Pressure 결측 처리: "해당 없음"을 0으로
#    - Student는 Work Pressure가 비어있는 게 정상 → 0
#    - Working은 Academic Pressure가 비어있는 게 정상 → 0
model_df['Work Pressure'] = model_df['Work Pressure'].fillna(0)
model_df['Academic Pressure'] = model_df['Academic Pressure'].fillna(0)

# 3) 나머지 결측은 드롭 (A단계: 최소 처리)
model_df = model_df.dropna()

print("원본 train_df:", train_df.shape)
print("모델용 model_df:", model_df.shape)

# 4) 더미화
model_df = pd.get_dummies(
    model_df,
    columns=[
        'Dietary Habits',
        'Have you ever had suicidal thoughts ?',
        'Working Professional or Student'
    ],
    drop_first=True
)

X = model_df.drop('Depression', axis=1)
y = model_df['Depression']

print("X shape:", X.shape, "y shape:", y.shape)

# 5) 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6) split + 학습
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("\n=== 로지스틱 회귀 계수(표준화) ===")
print(coef_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.barplot(
    data=coef_df,
    x='Coefficient',
    y='Feature',
    palette='coolwarm'
)

plt.axvline(0, color='black', linewidth=1)
plt.title("Logistic Regression Coefficients (Standardized)")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

In [ ]:
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_sorted = coef_df.sort_values(by='Abs_Coefficient', ascending=True)

plt.figure(figsize=(8,6))

sns.barplot(
    data=coef_sorted,
    x='Abs_Coefficient',
    y='Feature',
    palette='viridis'
)

plt.title("Variable Importance (Absolute Coefficient)")
plt.xlabel("Absolute Coefficient")
plt.ylabel("Feature")
plt.show()

1. Age (Coefficient = -2.02)
- 나이가 많아질수록 우울 위험 감소

2. Suicidal Thoughts (1.25)
- 원인이라기보단, 동반 지표로 해석하는 것이 맞을 것 같다.

3. Academic Pressure (1.21)
- 학생집단에서 매우 강력한 독립 요인.

4. Work Pressure (1.08)
- 직장인 내부에서 유의미하게 강함.

5. Financial Stress (0.77)

6. 식습관
- Unhealthy Diet (0.49)
- Moderate Diet (0.23)

-> 여전히 의미 있음.

7. Working Professional (-0.39)
- Student보다 상대적으로 위험이 낮음.

---

**가장 구조적인 취약 요인은 "청년층**

**그 다음 증폭 요인은 '압박과 스트레스'**

**생활습관(식습관)은 보조적인 요인**

**자살 생각 경험은 강력한 위험 신호**

## 전체 변수 기반

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# 0) Full 변수 후보
cols_full = [
    'Age',
    'Gender',
    'City',
    'Working Professional or Student',
    'Profession',
    'Academic Pressure',
    'Work Pressure',
    'CGPA',
    'Study Satisfaction',
    'Job Satisfaction',
    'Sleep Duration',
    'Dietary Habits',
    'Financial Stress',
    'Family History of Mental Illness',
    'Have you ever had suicidal thoughts ?',
    'Work/Study Hours',
    'Depression'
]

model_df = train_df[cols_full].copy()

# 1) 카테고리 정제: Dietary
valid_diet = ['Healthy', 'Moderate', 'Unhealthy']
model_df = model_df[model_df['Dietary Habits'].isin(valid_diet)]

# 2) 카테고리 정제: Sleep Duration (정상 4개만)
valid_sleep = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
model_df = model_df[model_df['Sleep Duration'].isin(valid_sleep)]

# 3) Profession / City는 상위 N + Other로 축소 (더미 폭발 방지)
top_prof = model_df['Profession'].value_counts().head(10).index
model_df['Profession'] = model_df['Profession'].where(model_df['Profession'].isin(top_prof), 'Other')

top_city = model_df['City'].value_counts().head(10).index
model_df['City'] = model_df['City'].where(model_df['City'].isin(top_city), 'Other')

# 4) "구조적 결측" 처리: 해당 없음은 0으로 채움
# Student는 Work Pressure, Job Satisfaction이 "해당 없음"
model_df['Work Pressure'] = model_df['Work Pressure'].fillna(0)
model_df['Job Satisfaction'] = model_df['Job Satisfaction'].fillna(0)

# Working은 Academic Pressure, Study Satisfaction, CGPA가 "해당 없음"
model_df['Academic Pressure'] = model_df['Academic Pressure'].fillna(0)
model_df['Study Satisfaction'] = model_df['Study Satisfaction'].fillna(0)
model_df['CGPA'] = model_df['CGPA'].fillna(0)

# 5) 나머지 결측만 제거 (극소)
model_df = model_df.dropna(subset=[
    'Age', 'Gender', 'City', 'Working Professional or Student',
    'Sleep Duration', 'Dietary Habits', 'Financial Stress',
    'Family History of Mental Illness', 'Have you ever had suicidal thoughts ?',
    'Work/Study Hours', 'Depression'
])

print("모델용 데이터 shape:", model_df.shape)

# 6) 더미화
model_df = pd.get_dummies(model_df, drop_first=True)

X = model_df.drop('Depression', axis=1)
y = model_df['Depression']

print("X shape:", X.shape, "y shape:", y.shape)
print("타겟 분포:\n", y.value_counts())

# 7) 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# stratify 가능한지 체크 (안전장치)
if y.value_counts().min() < 2:
    raise ValueError("한 클래스 표본이 2개 미만입니다. 전처리 필터가 너무 강합니다.")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=4000)
model.fit(X_train, y_train)

coef_df_full = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})
coef_df_full['Abs'] = coef_df_full['Coefficient'].abs()
coef_df_full = coef_df_full.sort_values('Abs', ascending=False)

print("\n=== 전체 변수 모델 Top 30 (절대값 기준) ===")
print(coef_df_full.head(30))

In [ ]:
top20 = coef_df_full.head(20).sort_values('Abs', ascending=True)

plt.figure(figsize=(9,6))
sns.barplot(data=top20, x='Abs', y='Feature')
plt.title("Top 20 Important Features (Full Model, |coef|)")
plt.xlabel("|Standardized Coefficient|")
plt.ylabel("")
plt.show()

1. Age (-2.05)
2. Suicidal Thoughts (1.25)
3. Academic Pressure (1.20)
4. Work Pressure (1.07)
5. Job Satisfaction (-0.89)
  - 만족도가 높을수록 보호 효과
  - 보호 변수 중 가장 강함.

6. Financial Stress (0.77)
- 청년층 분석과 일치.
7. Work/Study Hours (0.52)
- 장시간 공부/근무도 위험 요인.

---

- 식습관도 여전히 의미 있음.
- Study Satisfaction (-0.34)
  - 학생 집단 내 보호 효과,

- Profession / City
  - 효과 매우 작음.

  -> 직업 자체보단 '압박, 만족도, 연령'이 중요.

---

**학생과 직장인 집단을 분리하여 진행**

- 학생과 직장인은 구조가 다름을 확인함.
- Pressure 변수 등도 서로 다름.
- Satisfacation 도 위와 같음.
- CGPA는 학생에게만 의미가 있음.

-> 하나로 섞으면, 계수 왜곡 가능성이 있음

-> 분리해서 모델을 돌리고, 해석을 해야 더 깔끔한 분석이 가능해짐.

---

## 1. 학생 집단

- Academic Pressure

- Study Satisfaction

- CGPA

- Financial Stress

- Sleep Duration

- Dietary

- Suicidal Thoughts

- Work/Study Hours

- Age

- Gender

- Family History

- (Profession, City 제외)

In [ ]:
# Student만 추출
student_df = train_df[
    train_df['Working Professional or Student'] == 'Student'
].copy()

cols_student = [
    'Age',
    'Gender',
    'Academic Pressure',
    'Study Satisfaction',
    'CGPA',
    'Financial Stress',
    'Sleep Duration',
    'Dietary Habits',
    'Have you ever had suicidal thoughts ?',
    'Work/Study Hours',
    'Family History of Mental Illness',
    'Depression'
]

student_df = student_df[cols_student]

# Sleep 정제
valid_sleep = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
student_df = student_df[student_df['Sleep Duration'].isin(valid_sleep)]

# Dietary 정제
valid_diet = ['Healthy', 'Moderate', 'Unhealthy']
student_df = student_df[student_df['Dietary Habits'].isin(valid_diet)]

# 결측 처리
student_df = student_df.fillna(0)

print("Student 모델 데이터 shape:", student_df.shape)

student_df = pd.get_dummies(student_df, drop_first=True)

X_s = student_df.drop('Depression', axis=1)
y_s = student_df['Depression']

scaler = StandardScaler()
X_s_scaled = scaler.fit_transform(X_s)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s_scaled, y_s, test_size=0.2, random_state=42, stratify=y_s
)

model_s = LogisticRegression(max_iter=4000)
model_s.fit(X_train_s, y_train_s)

coef_student = pd.DataFrame({
    'Feature': X_s.columns,
    'Coefficient': model_s.coef_[0]
})

coef_student['Abs'] = coef_student['Coefficient'].abs()
coef_student = coef_student.sort_values('Abs', ascending=False)

print("\n=== Student 모델 중요 변수 ===")
print(coef_student.head(15))

In [ ]:
plt.figure(figsize=(8,6))
sns.barplot(
    data=coef_student.head(10).sort_values('Abs'),
    x='Abs',
    y='Feature'
)
plt.title("Top Student Risk Factors")
plt.show()

| 순위 | 변수                | 해석           |
| -- | ----------------- | ------------ |
| 1  | Suicidal Thoughts | 가장 강력한 경고 신호 |
| 2  | Academic Pressure | 핵심 구조 요인     |
| 3  | Financial Stress  | 중요한 증폭 요인    |
| 4  | Unhealthy Diet    | 생활 위험 요인     |
| 5  | Age (-)           | 나이 많을수록 보호   |

---

**Age 계수 (-0.52)**
- 전체 모델에서는 -2.5였는데, Student 집단에서는 -0.52로 약해짐.

=> Age 효과의 상당 부분은 "직장인 집단 포함 효과"였음.

**Academic Pressure 계수 (1.15)**
- 학생 집단에서 핵심 변수!

**Study Satisfaction (-0.33)**
- 만족도가 보호 역할을 한다.

**CGPA (0.08)**
- 거의 영향 없음.

**-> 성적 자체는 구조적 요인이 아님**


## 2. 직장인 집단

- Work Pressure

- Job Satisfaction

- Financial Stress

- Sleep Duration

- Dietary

- Suicidal Thoughts

- Work/Study Hours

- Age

- Gender

- Family History

- (Profession, City 제외)

In [ ]:
working_df = train_df[
    train_df['Working Professional or Student'] == 'Working Professional'
].copy()

cols_working = [
    'Age',
    'Gender',
    'Work Pressure',
    'Job Satisfaction',
    'Financial Stress',
    'Sleep Duration',
    'Dietary Habits',
    'Have you ever had suicidal thoughts ?',
    'Work/Study Hours',
    'Family History of Mental Illness',
    'Depression'
]

working_df = working_df[cols_working]

# Sleep 정제
working_df = working_df[working_df['Sleep Duration'].isin(valid_sleep)]

# Dietary 정제
working_df = working_df[working_df['Dietary Habits'].isin(valid_diet)]

working_df = working_df.fillna(0)

print("Working 모델 데이터 shape:", working_df.shape)

working_df = pd.get_dummies(working_df, drop_first=True)

X_w = working_df.drop('Depression', axis=1)
y_w = working_df['Depression']

X_w_scaled = scaler.fit_transform(X_w)

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_w_scaled, y_w, test_size=0.2, random_state=42, stratify=y_w
)

model_w = LogisticRegression(max_iter=4000)
model_w.fit(X_train_w, y_train_w)

coef_working = pd.DataFrame({
    'Feature': X_w.columns,
    'Coefficient': model_w.coef_[0]
})

coef_working['Abs'] = coef_working['Coefficient'].abs()
coef_working = coef_working.sort_values('Abs', ascending=False)

print("\n=== Working 모델 중요 변수 ===")
print(coef_working.head(15))

In [ ]:
plt.figure(figsize=(8,6))
sns.barplot(
    data=coef_working.head(10).sort_values('Abs'),
    x='Abs',
    y='Feature'
)
plt.title("Top Working Risk Factors")
plt.show()

| 순위 | 변수                       | 해석        |
| -- | ------------------------ | --------- |
| 1  | Age (-1.99)              | 압도적 구조 요인 |
| 2  | Suicidal Thoughts        | 강력 경고     |
| 3  | Work Pressure            | 핵심 위험 요인  |
| 4  | Financial Stress         | 증폭 요인     |
| 5  | Job Satisfaction (-0.75) | 강한 보호 요인  |

**Age 계수 (-1.99)**
- 특히 직장인 집단에서 Age가 매우 강함.
- 젊은 직장인 = 우울 위험 높음.
- 나이 들수록 안정.

**=> 경력 초기 스트레스 구조일 가능성 있음.**

**Job Satisfaction (-0.75)**
- 매우 강한 보호 요인



**Student 구조**

우울 위험은 “압박 + 재정 스트레스 + 생활습관” 구조

**Working 구조**

우울 위험은 “연령 + 직무 압박 + 직무 만족도” 구조

---

## 1. 청년층에서 우울 위험 비율이 높음.
## 2. 그런데 여성 청년층이 특히 더 높은가...??

### Age x Gender

In [ ]:
# Age Group 생성
df = train_df[['Age','Gender','Depression']].copy()

bins = [18,25,35,45,55,65]
labels = ['18-25','26-35','36-45','46-55','56-65']

df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

df = df.dropna()

In [ ]:
print("=== Age × Gender 분포 ===")
print(pd.crosstab(df['Age Group'], df['Gender']))

In [ ]:
rate_table = (
    df.groupby(['Age Group','Gender'])['Depression']
    .mean()
    .mul(100)
    .reset_index()
)

print("=== Age × Gender 우울 비율(%) ===")
print(rate_table)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.barplot(
    data=rate_table,
    x='Age Group',
    y='Depression',
    hue='Gender'
)

plt.title("Depression Rate by Age Group and Gender")
plt.ylabel("Depression Rate (%)")
plt.show()

In [ ]:
contingency = pd.crosstab(
    [df['Age Group'], df['Gender']],
    df['Depression']
)

from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency)

print("Chi-square:", chi2)
print("p-value:", p)

-> 이 데이터셋에서는 Gender 차이가 거의 없다.

-> 특히, 청년층에서도 여성 청년층이 특별히 더 높게 나오지 않는다.

---

### Q. 도메인 지식과 왜 다를까?

일반 연구에서는
- 여성의 우울 진단율이 더 높다.

이 데이터셋에서는
- 자가 진단? 설문 기반 데이터
- Depression 정의가 '우울 진단 확정'이 아닌, '우울증 위험도'이기에, 정의 방식이 달라서
- 표본 국가/문화 특성 영향 가능성 O
  - 인도와 우리나라의 차이..?

In [ ]:
train_df['City'].value_counts()

### 학생 집단 내부에서의 Gender 효과

#### Q. 학생 집단 안에서 여성 청년층이 더 높은가?

In [ ]:
student_df = train_df[
    train_df['Working Professional or Student'] == 'Student'
][['Age','Gender','Depression']].copy()

student_df['Age Group'] = pd.cut(
    student_df['Age'],
    bins=bins,
    labels=labels,
    right=False
)

student_df = student_df.dropna()

# 우울 비율
rate_student = (
    student_df.groupby(['Age Group','Gender'])['Depression']
    .mean()
    .mul(100)
    .reset_index()
)

print(rate_student)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(
    data=rate_student,
    x='Age Group',
    y='Depression',
    hue='Gender'
)

plt.title("Student: Depression Rate by Age & Gender")
plt.ylabel("Depression Rate (%)")
plt.show()

-> 학생 집단에서 Gender 효과는 거의 없다.

### Age × Gender × Academic Pressure

In [ ]:
student_df2 = train_df[
    train_df['Working Professional or Student'] == 'Student'
][['Age','Gender','Depression','Academic Pressure']].copy()

student_df2['Age Group'] = pd.cut(
    student_df2['Age'],
    bins=bins,
    labels=labels,
    right=False
)

# Pressure 이진화
student_df2['Pressure Level'] = np.where(
    student_df2['Academic Pressure'] >= 4,
    'High (4-5)',
    'Low/Medium (1-3)'
)

student_df2 = student_df2.dropna()

rate_pressure = (
    student_df2.groupby(['Age Group','Gender','Pressure Level'])['Depression']
    .mean()
    .mul(100)
    .reset_index()
)

print(rate_pressure)

In [ ]:
g = sns.catplot(
    data=rate_pressure,
    x='Age Group',
    y='Depression',
    hue='Gender',
    col='Pressure Level',
    kind='bar',
    height=4,
    aspect=1
)

g.fig.suptitle("Student: Age × Gender × Academic Pressure", y=1.05)
plt.show()

-> Gender 차이보다, Pressure 수준이 훨씬 강력한 결정 요인

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 0) Student 데이터만 준비 (Age, Gender, Depression, Academic Pressure)
student_df2 = train_df[
    train_df['Working Professional or Student'] == 'Student'
][['Age', 'Gender', 'Depression', 'Academic Pressure']].copy()

# 1) Age Group 생성
bins = [18, 25, 35, 45, 55, 65]
labels = ['18-25', '26-35', '36-45', '46-55', '56-65']
student_df2['Age Group'] = pd.cut(student_df2['Age'], bins=bins, labels=labels, right=False)

# 2) Pressure Level 생성 (High vs Low/Medium)
student_df2['Pressure Level'] = np.where(
    student_df2['Academic Pressure'] >= 4,
    'High (4-5)',
    'Low/Medium (1-3)'
)

# 3) 결측 제거 (Age Group, Academic Pressure 결측 제거)
student_df2 = student_df2.dropna(subset=['Age Group', 'Gender', 'Depression', 'Academic Pressure'])

# 4) statsmodels formula용 컬럼명 정리 (공백 제거)
student_df2 = student_df2.rename(columns={
    'Age Group': 'Age_Group',
    'Pressure Level': 'Pressure_Level'
})

# 5) 범주형 타입 지정
student_df2['Gender'] = student_df2['Gender'].astype('category')
student_df2['Age_Group'] = student_df2['Age_Group'].astype('category')
student_df2['Pressure_Level'] = student_df2['Pressure_Level'].astype('category')

# 6) 로지스틱 회귀 (interaction 포함)
model = smf.logit(
    'Depression ~ C(Age_Group) + C(Gender) + C(Pressure_Level)'
    ' + C(Age_Group):C(Gender)'
    ' + C(Gender):C(Pressure_Level)',
    data=student_df2
).fit()

print(model.summary())

#